In [1]:
from kiteconnect import KiteConnect
import pandas as pd
import datetime as dt
import os
import numpy as np
from stocktrends import Renko

In [2]:
request_token=open("request_token.txt","r").read()
key_secrets=open("api_key.txt","r").read().split()
kite = KiteConnect(api_key=key_secrets[0])
data = kite.generate_session(request_token=request_token,api_secret=key_secrets[1])

In [3]:
# Get dump of all NSE instruments
instruments_dump=kite.instruments("NSE")
instruments_df=pd.DataFrame(instruments_dump)
instruments_df.to_csv("NSE_Instruments.csv",index=False)

In [4]:
def instrumentLookup(instruments_df,symbol):
    """Looks up instrument token for a given script from instrument dump"""
    try:
        return instruments_df[instruments_df.tradingsymbol==symbol].instrument_token.values[0]
    except:
        return -1


def fetchOHLC(ticker,interval,duration):
    """extracts historical data and outputs in the form of dataframe"""
    instrument = instrumentLookup(instruments_df,ticker)
    data = pd.DataFrame(kite.historical_data(instrument,dt.date.today()-dt.timedelta(duration), dt.date.today(),interval))
    data.set_index("date",inplace=True)
    return data

In [5]:

def levels(ohlc_day):    
    """returns pivot point and support/resistance levels"""
    high = round(ohlc_day["high"][-1],2)
    low = round(ohlc_day["low"][-1],2)
    close = round(ohlc_day["close"][-1],2)
    pivot = round((high + low + close)/3,2)
    r1 = round((2*pivot - low),2)
    r2 = round((pivot + (high - low)),2)
    r3 = round((high + 2*(pivot - low)),2)
    s1 = round((2*pivot - high),2)
    s2 = round((pivot - (high - low)),2)
    s3 = round((low - 2*(high - pivot)),2)
    return (pivot,r1,r2,r3,s1,s2,s3)

In [6]:
ohlc_day = fetchOHLC("INFY","day",30)
pp_levels = levels(ohlc_day.iloc[:-1,:])

In [7]:
ohlc_day

,open,high,low,close,volume
date,,,,,
2025-03-13 00:00:00+05:30,1599.25,1606.05,1570.30,1579.85,7822888
2025-03-17 00:00:00+05:30,1545.15,1594.00,1545.15,1590.05,7526032
2025-03-18 00:00:00+05:30,1597.65,1612.90,1582.35,1609.35,6298054
2025-03-19 00:00:00+05:30,1603.00,1603.00,1572.80,1586.55,7387068
2025-03-20 00:00:00+05:30,1592.00,1631.90,1592.00,1615.55,7186750
2025-03-21 00:00:00+05:30,1577.95,1603.90,1563.65,1592.55,10074677
2025-03-24 00:00:00+05:30,1597.95,1607.10,1572.70,1592.75,8857726
2025-03-25 00:00:00+05:30,1605.00,1636.15,1605.00,1628.45,9890140
2025-03-26 00:00:00+05:30,1626.00,1637.75,1595.00,1599.45,6651302


In [8]:
pp_levels

(1398.35, 1414.8, 1425.9, 1442.35, 1387.25, 1370.8, 1359.7)